# Week 4 – Member 4
## Configuration with environment variables and pydantic-settings

**Teaching goal:** Explain how an application reads configuration from outside its Python code, validates the values, and keeps private settings out of GitHub.

**Suggested teaching time:** 20–30 minutes, followed by the separate 15-minute HTML quiz.

### Learning outcomes
By the end, your teammates should be able to:
1. Explain what application configuration is.
2. Explain why environment variables are useful.
3. Describe what `.env`, `.env.example`, and `.gitignore` do.
4. Create a settings class with `pydantic-settings`.
5. Explain required settings, defaults, and type validation.
6. Demonstrate how the Library API loads its settings.

## 1. Start with an everyday example

Imagine a library website. It needs to know:
- What name to display.
- Whether debug mode is on.
- Which port the application should use.
- Where its database is located.
- Which password or API key it should use.

These values are called **configuration**. They tell an application how to operate.

Ask your teammates: *Would a development copy and a live website always need the same database address and debug setting?* Usually not. That is one reason to keep configuration outside the Python source code.

## 2. Why not hardcode everything?

A hardcoded value is written directly into the program.

For example:

```python
database_password = "my-real-password"
```

This is risky because someone might upload the code to a shared or public GitHub repository. Different environments may also need different values.

A safer approach is to keep configuration outside the source code and let the application read it when it starts.

**Important:** An environment variable or `.env` file is not automatically secret-proof. Protect access to it and never publish real passwords or API keys.

## 3. What is an environment variable?

An **environment variable** is a named value supplied to a running program from its environment, rather than being written directly into the Python file.

On Ubuntu, in a terminal, you can set one like this:

```bash
export APP_NAME="My Library API"
```

You can inspect it with:

```bash
echo "$APP_NAME"
```

Python can read it with the built-in `os` module:

In [ ]:
import os

app_name = os.getenv("APP_NAME", "Library API")
print(app_name)

`os.getenv("APP_NAME")` reads the variable named `APP_NAME`. If it is missing and no default is supplied, it returns `None`.

With `os.getenv("APP_NAME", "Library API")`, Python uses `"Library API"` as a fallback when the variable is missing.

**Terminal note:** An `export` normally applies to that terminal session and processes launched from it. A VS Code or Jupyter process opened earlier may not inherit a variable you export later.

## 4. What is a `.env` file?

A `.env` file is a simple text file containing configuration values as `NAME=value` pairs.

Example for the Library API:

```text
APP_NAME=My Library API
DEBUG=true
PORT=8000
DATABASE_URL=sqlite:///library.db
```

The `.env` file is a convenient way to provide values locally. A library such as `pydantic-settings` can be configured to read it.

Do not add real passwords or API keys to screenshots, notebooks, or public repositories.

## 5. Pydantic and pydantic-settings

- **Pydantic** helps validate data and convert values to the types your Python application expects.
- **pydantic-settings** builds on Pydantic to load and validate application settings, including values from environment variables and (when configured) a `.env` file.
- **`BaseSettings`** is the class your settings class inherits from.

Install the packages in your project's active virtual environment:

```bash
python -m pip install pydantic-settings python-dotenv
```

`python-dotenv` is commonly used to support reading `.env` files. If your project already has dependencies installed, check your existing `requirements.txt` first.

## 6. Build the settings class

Create a file named `config.py`:

```python
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    app_name: str = "Library API"
    debug: bool = False
    port: int = 8000
    database_url: str

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8"
    )

settings = Settings()
```

Understand each part:

| Code | Meaning |
|---|---|
| `class Settings(BaseSettings)` | Defines a settings class that loads and validates configuration. |
| `app_name: str` | `app_name` should be text. |
| `= "Library API"` | Default value used if no external value is supplied. |
| `debug: bool = False` | Boolean setting; defaults to off. |
| `port: int = 8000` | Integer setting; defaults to 8000. |
| `database_url: str` | Required text setting because it has no default. |
| `env_file=".env"` | Tells the settings class to read the `.env` file. |
| `settings = Settings()` | Creates the settings object and loads/validates the values. |

By default, pydantic-settings maps names such as `app_name` to environment variable names such as `APP_NAME`. It can parse values such as `8000` into an integer and `true` into a Boolean.

## 7. Try the Library API settings

Make sure your `.env` file is in the same project directory from which the application is run (or configure the path explicitly).

Example `.env`:

```text
APP_NAME=My Library API
DEBUG=true
PORT=8000
DATABASE_URL=sqlite:///library.db
```

In `main.py`:

```python
from config import settings

print(settings.app_name)
print(settings.debug)
print(settings.port)
print(settings.database_url)
```

Expected output:

```text
My Library API
True
8000
sqlite:///library.db
```

The output shows the values your application loaded. In a real project, do not print passwords, tokens, or other secrets.

In [ ]:
# Optional demonstration cell.
# Run this after creating config.py and .env in the same project directory.

from config import settings

print(settings.app_name)
print(settings.debug)
print(settings.port)
print(settings.database_url)

## 8. Defaults, required values, and validation

### A setting with a default

```python
port: int = 8000
```

If no `PORT` value is supplied, the default is 8000.

### A required setting

```python
database_url: str
```

There is no default. The application must receive a database URL from a configured source, such as the environment or `.env`. If it is missing, settings validation fails.

### Type validation

If the setting expects an integer:

```python
port: int = 8000
```

and the supplied value is `PORT=hello`, Pydantic cannot convert `"hello"` into an integer and raises a validation error.

This is useful because the application can catch a bad configuration when it starts instead of discovering it later during a request.

## 9. Protect the real `.env` file with Git

Create a `.gitignore` file in the project root:

```text
.env
venv/
.venv/
__pycache__/
```

This tells Git not to track these paths when they are untracked.

**Important limitation:** `.gitignore` does not remove a secret that has already been committed or pushed. If a real credential is exposed, revoke or rotate it and follow your team's repository cleanup process.

### What about teammates?

Create a safe `.env.example` file with placeholder values:

```text
APP_NAME=My Library API
DEBUG=false
PORT=8000
DATABASE_URL=your_database_url_here
```

Usually:
- `.env` — keep local and do not commit.
- `.env.example` — commit as a guide, with placeholders only.
- `.gitignore` — commit so Git knows what to ignore.
- `config.py` and `main.py` — commit, provided they contain no secrets.

## 10. Development versus production

A development setup might use:
- `DEBUG=true`
- A local test database.

A production setup might use:
- `DEBUG=false`
- A production database.
- Credentials supplied through the hosting platform's environment or a dedicated secrets manager.

Do not use real production credentials in a class demo. Use harmless example values.

## 11. Common misunderstandings

| Do not confuse | Difference |
|---|---|
| `.venv` and environment variables | `.venv` isolates Python packages; environment variables supply application settings. |
| `.env` and `.gitignore` | `.env` stores values; `.gitignore` tells Git not to track specified files. |
| Pydantic and pydantic-settings | Pydantic validates data; pydantic-settings loads and validates application settings. |
| Default and required setting | A default can be used when a value is absent; a required setting must be supplied. |
| Hiding a file and securing a secret | Ignoring a file helps prevent accidental commits, but does not secure a leaked or already committed secret. |

## 12. A simple teaching demonstration

Use this order when presenting to your teammates:

1. **Introduce the problem:** An application needs different settings in different environments.
2. **Show an environment variable:** Demonstrate `export APP_NAME="My Library API"` and `os.getenv`.
3. **Show `.env`:** Explain the `NAME=value` format.
4. **Show `config.py`:** Explain `BaseSettings`, types, defaults, and required values.
5. **Run `main.py`:** Show the four expected Library API outputs.
6. **Demonstrate validation:** Temporarily change `PORT=8000` to `PORT=hello`, run again, and explain the validation error. Restore `PORT=8000` afterwards.
7. **Explain Git safety:** Show `.gitignore` and `.env.example`.

### Suggested opening

“Good morning, everyone. My topic is configuration with environment variables and pydantic-settings. Applications need information such as a database address, port, and debug setting. Instead of writing these values directly into our Python code, we can supply them from the environment. pydantic-settings helps our application load and validate those values.”

## 13. Check your understanding

Try answering these aloud before looking at the separate HTML quiz.

1. What does configuration mean in an application?
2. Why is hardcoding a real password risky?
3. What is an environment variable?
4. What does `os.getenv()` do?
5. What is the purpose of a `.env` file?
6. What does `BaseSettings` provide?
7. What is the difference between `port: int = 8000` and `database_url: str`?
8. What happens if `PORT=hello`?
9. Why should `.env` be listed in `.gitignore`?
10. Why is `.env.example` useful for teammates?

### Friday mini-project checklist

- [ ] Create a `config.py` settings class using `pydantic-settings`.
- [ ] Load settings from environment variables and a `.env` file.
- [ ] Include sensible defaults where appropriate.
- [ ] Include required settings where appropriate.
- [ ] Demonstrate that incorrect types cause validation errors.
- [ ] Add `.env` and virtual-environment folders to `.gitignore`.
- [ ] Add `.env.example` with placeholders, not real secrets.
- [ ] Document each configuration variable and whether it is required or has a default.
- [ ] Confirm no real passwords, API keys, or tokens are committed.

**Teaching reminder:** Explain the idea first, show the code second, and ask a teammate to explain the result in their own words.